In [1]:
import os
import pandas as pd
from datasets import load_dataset

print("Downloading...")

try:
    sciq_dataset = load_dataset("allenai/sciq")
    save_path = "../datasets/sciq_train.csv"
    train_df = pd.DataFrame(sciq_dataset["train"])
    train_df.to_csv(save_path, index=False)

    print("The data set was saved successfully!")
    print("Total number of questions ")
    
    
    display(train_df.head(3))

except Exception as e:
    print("Error")

Downloading...


The data set was saved successfully!
Total number of questions 


,question,distractor3,distractor1,distractor2,correct_answer,support
0,What type of organism is commonly used in prep...,viruses,protozoa,gymnosperms,mesophilic organisms,"Mesophiles grow best in moderate temperature, ..."
1,What phenomenon makes global winds blow northe...,tropical effect,muon effect,centrifugal effect,coriolis effect,Without Coriolis Effect the global winds would...
2,Changes from a less-ordered state to a more-or...,endothermic,unbalanced,reactive,exothermic,Summary Changes of state are examples of phase...


In [2]:
import sys
from transformers import T5Tokenizer, T5ForConditionalGeneration

print("T5-Small model is loading from Hugging Face...")

try:
    
    tokenizer = T5Tokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")
    
    text = "Photosynthesis is the process by which plants produce food."
    input_text = f"generate question: {text}"
    
    inputs = tokenizer(input_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=50, num_beams=4, early_stopping=True)
    generated_question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print("-" * 50)
    print(f"First Sentence (Input): {text}")
    print(f"AI Generated Question (Output): {generated_question}")
    print("-" * 50)
    print("Success! The pre-trained model works!")

except Exception as e:
    print(" Error occurred")

T5-Small model is loading from Hugging Face...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

--------------------------------------------------
First Sentence (Input): Photosynthesis is the process by which plants produce food.
AI Generated Question (Output): plants produce food.
--------------------------------------------------
Success! The pre-trained model works!


In [3]:

print("The columns in the dataset :")
print(train_df.columns.tolist())
print("-" * 70)


print("The first 5 rows of the dataset (Sample Data) :")
display(train_df.head())

The columns in the dataset :
['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support']
----------------------------------------------------------------------
The first 5 rows of the dataset (Sample Data) :


,question,distractor3,distractor1,distractor2,correct_answer,support
0,What type of organism is commonly used in prep...,viruses,protozoa,gymnosperms,mesophilic organisms,"Mesophiles grow best in moderate temperature, ..."
1,What phenomenon makes global winds blow northe...,tropical effect,muon effect,centrifugal effect,coriolis effect,Without Coriolis Effect the global winds would...
2,Changes from a less-ordered state to a more-or...,endothermic,unbalanced,reactive,exothermic,Summary Changes of state are examples of phase...
3,What is the least dangerous radioactive decay?,zeta decay,beta decay,gamma decay,alpha decay,All radioactive decay is dangerous to living t...
4,Kilauea in hawaii is the world’s most continuo...,magma,greenhouse gases,carbon and smog,smoke and ash,Example 3.5 Calculating Projectile Motion: Hot...


In [4]:
# STEP 1: Preparing the data to fit the T5 model
train_df["input_text"] = "generate question: " + train_df["support"]
train_df["target_text"] = train_df["question"]
train_df = train_df[["input_text", "target_text"]]

print(" Data Formatting Successful.\n")

# STEP 2: Viewing the total number of rows
print(f" Total Rows: {len(train_df)}")
print("-" * 50)

# STEP 3: Train/Test Split (90% Train, 10% Test)
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42
)

print(f" Train Data rows: {len(train_data)}")
print(f"Test Data rows: {len(test_data)}")
print("-" * 50)

# Check the shape of the new data structure
display(train_data.head(2))

 Data Formatting Successful.

 Total Rows: 11679
--------------------------------------------------
 Train Data rows: 10511
Test Data rows: 1168
--------------------------------------------------


,input_text,target_text
4684,generate question: A familiar liquid is mercur...,What is the only metal that is liquid at room ...
960,generate question: oily substance produced in ...,What kind of gland produces an oily substance ...


In [5]:
import torch
print(torch.__version__)

2.12.0+cpu


In [6]:

print(" Dataset Shape (Rows, Columns):", train_df.shape)
print("-" * 50)


train_df_small = train_data.head(1000)
test_df_small = test_data.head(100)

print(f" Data allocation for Small Prototype successful!")
print(f" Samples for Training: {len(train_df_small)}")
print(f"Samples for testing: {len(test_df_small)}")

 Dataset Shape (Rows, Columns): (11679, 2)
--------------------------------------------------
 Data allocation for Small Prototype successful!
 Samples for Training: 1000
Samples for testing: 100


In [7]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import Dataset

print(" Loading T5 Tokenizer and preparing datasets...")


model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# 2. Turning our Pandas DataFrames into Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df_small)
test_dataset = Dataset.from_pandas(test_df_small)

# 3. Function that converts text into numbers (tokens)
def preprocess_function(examples):
   # Tokenizing the input text (support text)
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True, padding="max_length")
    
   # Tokenizing the target text (question)
    labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True, padding="max_length")
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 4. Applying Tokenization to the Entire Dataset
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

print("\n Tokenization Successfully Completed for the Prototype!")
print(f"Train Dataset Features: {tokenized_train.column_names}")

 Loading T5 Tokenizer and preparing datasets...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


 Tokenization Successfully Completed for the Prototype!
Train Dataset Features: ['input_text', 'target_text', '__index_level_0__', 'input_ids', 'attention_mask', 'labels']
